# Data Quality Assessment: Input Data Issues & Impact**Business Question:** What data quality issues exist in the raw input files, and what impact do they have on downstream analysis?**Data Source:** Silver layer (cleaned & flagged data from `data/silver/*.parquet`) + pre-computed DQ log.**Output:**- Interactive DQ scorecard (overview of all issues)- Deep-dive charts per issue category- Combined impact summary- DQ report export

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
import vl_convert as vlc

# Color palette (Helbling style)
BLUE = "#1f4e79"
LBLUE = "#5b9bd5"
ORANGE = "#ed7d31"
GREY = "#a6a6a6"
GREEN = "#2e7d32"
RED = "#c0392b"
LIGHT_GREEN = "#e2efda"
LIGHT_RED = "#f8dcdb"
LIGHT_YELLOW = "#fff2cc"
LIGHT_BLUE = "#e7f0f7"

# Paths
NOTEBOOK_DIR = Path(os.getcwd())
REPO = NOTEBOOK_DIR.parent
SILVER_DIR = REPO / 'data' / 'silver'
CUSTOMER_SILVER = SILVER_DIR / 'customer.parquet'
PRODUCT_SILVER = SILVER_DIR / 'product.parquet'
DQ_LOG = SILVER_DIR / '_dq_log.csv'
EXPORT_DIR = NOTEBOOK_DIR

print(f"Repository: {REPO}")
print(f"Silver data: {SILVER_DIR}")
print(f"Export to: {EXPORT_DIR}")

## 2. Load Data from Silver Layer

In [ ]:
# Load data from silver layer
if not CUSTOMER_SILVER.exists():
    raise FileNotFoundError(f"Silver layer data not found: {CUSTOMER_SILVER}\nRun: python src/pipeline.py")
if not PRODUCT_SILVER.exists():
    raise FileNotFoundError(f"Silver layer data not found: {PRODUCT_SILVER}\nRun: python src/pipeline.py")
if not DQ_LOG.exists():
    raise FileNotFoundError(f"DQ log not found: {DQ_LOG}\nRun: python src/build_silver.py")

print(f"Loading silver data...")
customer_data = pd.read_parquet(CUSTOMER_SILVER)
product_data = pd.read_parquet(PRODUCT_SILVER)
dq_log = pd.read_csv(DQ_LOG)

print(f"\nCustomer data: {len(customer_data):,} transaction rows, {customer_data.shape[1]} columns")
print(f"Product data: {len(product_data):,} transaction rows, {product_data.shape[1]} columns")
print(f"DQ log: {len(dq_log)} checks")

print(f"\nCustomer columns: {customer_data.columns.tolist()}")
print(f"\nProduct columns: {product_data.columns.tolist()}")

## 3. Data Quality Scorecard (Overview)

All 10 DQ issues at a glance, colored by severity (% of rows affected).

In [ ]:
# Build scorecard from DQ logscorecard = dq_log.copy()scorecard['value_num'] = pd.to_numeric(scorecard['value'], errors='coerce')# Compute percentages - use a list and create new dataframe to avoid setitem issuespct_list = []for idx, row in scorecard.iterrows():    val = row['value_num']    if pd.notna(val) and val > 0:        if 'customer' in row['table'].lower():            total_rows = len(customer_data)        else:            total_rows = len(product_data)        pct = (val / total_rows) * 100        pct_list.append(pct)    else:        pct_list.append(0.0)scorecard['pct_affected'] = pct_listprint("DQ Scorecard:")print(scorecard[['table', 'check', 'value', 'pct_affected']].to_string())# Create scorecard summary for charting (exclude meta-checks)scorecard_chart = []skip_checks = ['rows', 'exact_duplicate_rows_dropped', 'customer_dimension_mapped', 'customer_groups_net_negative_gp']for idx, row in scorecard.iterrows():    if row['check'] in skip_checks:        continue        table_name = row['table'].capitalize()    pct = row['pct_affected']    count = int(row['value_num']) if pd.notna(row['value_num']) else 0        if count > 0:        severity = 'High' if pct > 10 else ('Medium' if pct > 5 else 'Low')        scorecard_chart.append({            'check': row['check'],            'table': table_name,            'pct_affected': pct,            'count': count,            'severity': severity        })scorecard_df = pd.DataFrame(scorecard_chart)print(f"Scorecard for charting ({len(scorecard_df)} issues):")print(scorecard_df.to_string())

In [ ]:
# Build scorecard from DQ log
scorecard = dq_log.copy()
scorecard['value_num'] = pd.to_numeric(scorecard['value'], errors='coerce')

# Compute percentages
scorecard['pct_affected'] = 0.0
for idx, row in scorecard.iterrows():
    val = row['value_num']
    if pd.notna(val) and val > 0:
        if 'customer' in row['table'].lower():
            total_rows = len(customer_data)
        else:
            total_rows = len(product_data)
        pct = (val / total_rows) * 100
        scorecard.loc[idx, 'pct_affected'] = pct

print("DQ Scorecard:")
print(scorecard[['table', 'check', 'value', 'pct_affected']].to_string())

# Create scorecard summary for charting (exclude meta-checks)
scorecard_chart = []
skip_checks = ['rows', 'exact_duplicate_rows_dropped', 'customer_dimension_mapped', 'customer_groups_net_negative_gp']

for idx, row in scorecard.iterrows():
    if row['check'] in skip_checks:
        continue
    
    table_name = row['table'].capitalize()
    pct = row['pct_affected']
    count = int(row['value_num']) if pd.notna(row['value_num']) else 0
    
    if count > 0:
        severity = 'High' if pct > 10 else ('Medium' if pct > 5 else 'Low')
        scorecard_chart.append({
            'check': row['check'],
            'table': table_name,
            'pct_affected': pct,
            'count': count,
            'severity': severity
        })

scorecard_df = pd.DataFrame(scorecard_chart)
print(f"\nScorecard for charting ({len(scorecard_df)} issues):")
print(scorecard_df.to_string())

## 4. Missing Data Deep Dives

### 4a. Buying Group L6 Missing (Product)

In [ ]:
# Check: is_missing_buying_group on product
if 'is_missing_buying_group' in product_data.columns:
    missing_bg_count = product_data['is_missing_buying_group'].sum()
    missing_bg_pct = (missing_bg_count / len(product_data)) * 100
    
    print(f"Product rows missing buying_group_l6: {missing_bg_count:,} ({missing_bg_pct:.1f}%)")
    print(f"Expected from PDF: 2,009 (5.4%)")
    print(f"Match: {abs(missing_bg_pct - 5.4) < 0.5}")
    
    # Create pie/donut data
    pie_data = [
        {"category": "Has Buying Group", "count": len(product_data) - missing_bg_count},
        {"category": "Missing Buying Group", "count": missing_bg_count}
    ]
    
    spec_pie_bg = {
        "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
        "title": "Product Rows: Buying Group L6 Coverage",
        "width": 400,
        "height": 300,
        "data": {"values": pie_data},
        "mark": {"type": "arc", "innerRadius": 50},
        "encoding": {
            "theta": {"field": "count", "type": "quantitative"},
            "color": {
                "field": "category",
                "type": "nominal",
                "scale": {"domain": ["Has Buying Group", "Missing Buying Group"], "range": [GREEN, RED]}
            },
            "tooltip": [
                {"field": "category", "type": "nominal", "title": "Status"},
                {"field": "count", "type": "quantitative", "title": "Row Count"}
            ]
        }
    }
    
    html_pie_bg = vlc.vegalite_to_html(vl_spec=json.dumps(spec_pie_bg))
    with open(EXPORT_DIR / 'dq_missing_buying_group.html', 'w') as f:
        f.write(html_pie_bg)
    print("✓ Chart saved to dq_missing_buying_group.html")
    display(HTML(html_pie_bg))
else:
    print("WARNING: 'is_missing_buying_group' column not found in product data")

### 4b. Missing Gross Profit (Product)

In [ ]:
# Check: is_missing_gp on product
if 'is_missing_gp' in product_data.columns:
    missing_gp_count = product_data['is_missing_gp'].sum()
    missing_gp_pct = (missing_gp_count / len(product_data)) * 100
    
    print(f"Product rows missing gross_profit: {missing_gp_count:,} ({missing_gp_pct:.1f}%)")
    print(f"Expected from PDF: 3,889 (10.5%)")
    print(f"Match: {abs(missing_gp_pct - 10.5) < 0.5}")
    
    # Stacked bar by business_unit
    missing_by_bu = product_data.groupby('business_unit').agg({
        'is_missing_gp': 'sum',
        'business_unit': 'size'
    }).rename(columns={'business_unit': 'total'}).reset_index()
    missing_by_bu.columns = ['business_unit', 'missing', 'total']
    missing_by_bu['has_gp'] = missing_by_bu['total'] - missing_by_bu['missing']
    
    # Reshape for stacked bar
    chart_data = []
    for idx, row in missing_by_bu.iterrows():
        chart_data.append({"business_unit": row['business_unit'], "status": "Has GP", "count": row['has_gp']})
        chart_data.append({"business_unit": row['business_unit'], "status": "Missing GP", "count": row['missing']})
    
    spec_missing_gp = {
        "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
        "title": "Gross Profit Completeness by Business Unit",
        "width": 500,
        "height": 300,
        "data": {"values": chart_data},
        "mark": "bar",
        "encoding": {
            "x": {"field": "business_unit", "type": "nominal", "title": "Business Unit"},
            "y": {"field": "count", "type": "quantitative", "title": "Row Count"},
            "color": {
                "field": "status",
                "type": "nominal",
                "scale": {"domain": ["Has GP", "Missing GP"], "range": [GREEN, RED]}
            }
        }
    }
    
    html_gp = vlc.vegalite_to_html(vl_spec=json.dumps(spec_missing_gp))
    with open(EXPORT_DIR / 'dq_missing_gp.html', 'w') as f:
        f.write(html_gp)
    print("✓ Chart saved to dq_missing_gp.html")
    display(HTML(html_gp))
else:
    print("WARNING: 'is_missing_gp' column not found in product data")

### 4c. Missing Booked Date (Customer)

In [ ]:
# Check: missing booked_date
missing_date = customer_data['booked_date'].isna().sum()
missing_date_pct = (missing_date / len(customer_data)) * 100

print(f"Customer rows missing booked_date: {missing_date:,} ({missing_date_pct:.1f}%)")
print(f"Expected from PDF: 23,252 (20.1%)")
print(f"Match: {abs(missing_date_pct - 20.1) < 0.5}")

# Distribution by year (for rows with valid booked_date)
df_with_date = customer_data[customer_data['booked_date'].notna()].copy()
df_with_date['date_year'] = df_with_date['booked_date'].dt.year
year_dist = df_with_date['date_year'].value_counts().sort_index().reset_index()
year_dist.columns = ['year', 'rows_with_date']

# Also add rows without date
years_with_missing = customer_data['year'].unique()
chart_data_dates = []
for y in sorted(years_with_missing):
    rows_year = len(customer_data[customer_data['year'] == y])
    rows_with_date_year = len(customer_data[(customer_data['year'] == y) & (customer_data['booked_date'].notna())])
    rows_missing_date_year = rows_year - rows_with_date_year
    chart_data_dates.append({"year": str(int(y)), "status": "Has Booked Date", "count": rows_with_date_year})
    if rows_missing_date_year > 0:
        chart_data_dates.append({"year": str(int(y)), "status": "Missing Booked Date", "count": rows_missing_date_year})

spec_dates = {
    "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
    "title": "Booked Date Completeness by Year (Customer Data)",
    "width": 600,
    "height": 300,
    "data": {"values": chart_data_dates},
    "mark": "bar",
    "encoding": {
        "x": {"field": "year", "type": "ordinal", "title": "Year"},
        "y": {"field": "count", "type": "quantitative", "title": "Row Count"},
        "color": {
            "field": "status",
            "type": "nominal",
            "scale": {"domain": ["Has Booked Date", "Missing Booked Date"], "range": [GREEN, RED]}
        }
    }
}

html_dates = vlc.vegalite_to_html(vl_spec=json.dumps(spec_dates))
with open(EXPORT_DIR / 'dq_missing_dates.html', 'w') as f:
    f.write(html_dates)
print("✓ Chart saved to dq_missing_dates.html")
display(HTML(html_dates))

### 4d. Coverage Gap (Date Range Mismatch)

In [ ]:
# Coverage gap: product 2021-2026, customer 2025-2026
cust_min_year = customer_data['year'].min()
cust_max_year = customer_data['year'].max()
prod_min_year = product_data['year'].min() if 'year' in product_data.columns else product_data[product_data.columns[product_data.columns.str.contains('year', case=False)]].min().values[0] if any(product_data.columns.str.contains('year', case=False)) else None
prod_max_year = product_data['year'].max() if 'year' in product_data.columns else product_data[product_data.columns[product_data.columns.str.contains('year', case=False)]].max().values[0] if any(product_data.columns.str.contains('year', case=False)) else None

print(f"Customer data year range: {int(cust_min_year)} - {int(cust_max_year)}")
if prod_min_year is not None and prod_max_year is not None:
    print(f"Product data year range: {int(prod_min_year)} - {int(prod_max_year)}")
    overlap_min = max(cust_min_year, prod_min_year)
    overlap_max = min(cust_max_year, prod_max_year)
    print(f"Overlap period: {int(overlap_min)} - {int(overlap_max)}")
    
    # Create timeline visualization
    timeline_data = [
        {"dataset": "Customer View", "start": int(cust_min_year), "end": int(cust_max_year), "label": f"Customer: {int(cust_min_year)}-{int(cust_max_year)}"},
        {"dataset": "Product View", "start": int(prod_min_year), "end": int(prod_max_year), "label": f"Product: {int(prod_min_year)}-{int(prod_max_year)}"}
    ]
    
    spec_coverage = {
        "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
        "title": "Data Coverage Gap: Year Range Mismatch",
        "width": 600,
        "height": 150,
        "data": {"values": timeline_data},
        "mark": "bar",
        "encoding": {
            "y": {"field": "dataset", "type": "nominal", "title": ""},
            "x": {"field": "start", "type": "quantitative", "title": "Year"},
            "x2": {"field": "end"},
            "color": {"field": "dataset", "type": "nominal", "scale": {"domain": ["Customer View", "Product View"], "range": [ORANGE, BLUE]}, "legend": None},
            "tooltip": [{"field": "label", "type": "nominal", "title": "Coverage"}]
        }
    }
    
    html_coverage = vlc.vegalite_to_html(vl_spec=json.dumps(spec_coverage))
    with open(EXPORT_DIR / 'dq_coverage_gap.html', 'w') as f:
        f.write(html_coverage)
    print("✓ Chart saved to dq_coverage_gap.html")
    display(HTML(html_coverage))
else:
    print("Note: Could not extract year range from product data")

## 5. Faulty / Implausible Data

### 5a. Negative Gross Profit Lines

In [ ]:
# Negative GP on both tables
if 'is_negative_gp' in customer_data.columns and 'is_negative_gp' in product_data.columns:
    cust_neg_gp = customer_data['is_negative_gp'].sum()
    cust_neg_gp_pct = (cust_neg_gp / len(customer_data)) * 100
    prod_neg_gp = product_data['is_negative_gp'].sum()
    prod_neg_gp_pct = (prod_neg_gp / len(product_data)) * 100
    
    print(f"Customer rows with negative GP: {cust_neg_gp:,} ({cust_neg_gp_pct:.1f}%)")
    print(f"  Expected from PDF: 14,259 (12.3%)")
    print(f"Product rows with negative GP: {prod_neg_gp:,} ({prod_neg_gp_pct:.1f}%)")
    print(f"  Expected from PDF: 3,235 (8.8%)")
    
    # Side-by-side comparison
    neg_gp_data = [
        {"table": "Customer", "status": "Positive GP", "count": len(customer_data) - cust_neg_gp},
        {"table": "Customer", "status": "Negative GP", "count": cust_neg_gp},
        {"table": "Product", "status": "Positive GP", "count": len(product_data) - prod_neg_gp},
        {"table": "Product", "status": "Negative GP", "count": prod_neg_gp}
    ]
    
    spec_neg_gp = {
        "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
        "title": "Negative Gross Profit Lines: Customer vs Product",
        "width": 500,
        "height": 300,
        "data": {"values": neg_gp_data},
        "mark": "bar",
        "encoding": {
            "x": {"field": "table", "type": "nominal", "title": "Data Table"},
            "y": {"field": "count", "type": "quantitative", "title": "Row Count"},
            "color": {
                "field": "status",
                "type": "nominal",
                "scale": {"domain": ["Positive GP", "Negative GP"], "range": [GREEN, RED]}
            },
            "tooltip": [
                {"field": "table", "type": "nominal", "title": "Table"},
                {"field": "status", "type": "nominal", "title": "Status"},
                {"field": "count", "type": "quantitative", "title": "Count"}
            ]
        }
    }
    
    html_neg_gp = vlc.vegalite_to_html(vl_spec=json.dumps(spec_neg_gp))
    with open(EXPORT_DIR / 'dq_negative_gp.html', 'w') as f:
        f.write(html_neg_gp)
    print("✓ Chart saved to dq_negative_gp.html")
    display(HTML(html_neg_gp))
else:
    print("WARNING: 'is_negative_gp' column not found")

### 5b. Cost Without Sales (net_sales=0, GP≠0)

In [ ]:
# Cost without sales on both tables
if 'is_cost_without_sales' in customer_data.columns and 'is_cost_without_sales' in product_data.columns:
    cust_cws = customer_data['is_cost_without_sales'].sum()
    cust_cws_pct = (cust_cws / len(customer_data)) * 100
    prod_cws = product_data['is_cost_without_sales'].sum()
    prod_cws_pct = (prod_cws / len(product_data)) * 100
    
    print(f"Customer rows with cost-without-sales: {cust_cws:,} ({cust_cws_pct:.1f}%)")
    print(f"  Expected from PDF: 3,844")
    print(f"Product rows with cost-without-sales: {prod_cws:,} ({prod_cws_pct:.1f}%)")
    print(f"  Expected from PDF: 692")
    
    cws_data = [
        {"table": "Customer", "status": "Normal", "count": len(customer_data) - cust_cws},
        {"table": "Customer", "status": "Cost-Without-Sales", "count": cust_cws},
        {"table": "Product", "status": "Normal", "count": len(product_data) - prod_cws},
        {"table": "Product", "status": "Cost-Without-Sales", "count": prod_cws}
    ]
    
    spec_cws = {
        "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
        "title": "Anomalous Booking Pattern: Cost Without Sales",
        "width": 500,
        "height": 300,
        "data": {"values": cws_data},
        "mark": "bar",
        "encoding": {
            "x": {"field": "table", "type": "nominal", "title": "Data Table"},
            "y": {"field": "count", "type": "quantitative", "title": "Row Count"},
            "color": {
                "field": "status",
                "type": "nominal",
                "scale": {"domain": ["Normal", "Cost-Without-Sales"], "range": [GREEN, ORANGE]}
            }
        }
    }
    
    html_cws = vlc.vegalite_to_html(vl_spec=json.dumps(spec_cws))
    with open(EXPORT_DIR / 'dq_cost_without_sales.html', 'w') as f:
        f.write(html_cws)
    print("✓ Chart saved to dq_cost_without_sales.html")
    display(HTML(html_cws))
else:
    print("WARNING: 'is_cost_without_sales' column not found")

### 5c. Zero / Negative Sales (Returns, Credits, Rebates)

In [ ]:
# Zero/negative sales (customer only)
if 'is_zero_sales' in customer_data.columns:
    cust_zero_sales = customer_data['is_zero_sales'].sum()
    cust_zero_sales_pct = (cust_zero_sales / len(customer_data)) * 100
    
    print(f"Customer rows with zero/negative sales: {cust_zero_sales:,} ({cust_zero_sales_pct:.1f}%)")
    print(f"  Expected from PDF: 26,629 (23%)")
    
    # Distribution by region
    zero_sales_by_region = customer_data.groupby('region').agg({
        'is_zero_sales': 'sum',
        'region': 'size'
    }).rename(columns={'region': 'total'}).reset_index()
    zero_sales_by_region.columns = ['region', 'zero_sales_count', 'total']
    zero_sales_by_region['zero_sales_pct'] = (zero_sales_by_region['zero_sales_count'] / zero_sales_by_region['total']) * 100
    zero_sales_by_region = zero_sales_by_region.sort_values('zero_sales_count', ascending=False)
    
    spec_zero_sales = {
        "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
        "title": "Zero/Negative Sales by Region (Returns, Credits, Rebates)",
        "width": 600,
        "height": 300,
        "data": {"values": zero_sales_by_region.to_dict(orient='records')},
        "mark": "bar",
        "encoding": {
            "x": {"field": "region", "type": "nominal", "title": "Region", "sort": "-y"},
            "y": {"field": "zero_sales_count", "type": "quantitative", "title": "Row Count (Returns/Credits)"},
            "color": {"value": ORANGE},
            "tooltip": [
                {"field": "region", "type": "nominal", "title": "Region"},
                {"field": "zero_sales_count", "type": "quantitative", "title": "Count"},
                {"field": "zero_sales_pct", "type": "quantitative", "title": "% of Region", "format": ".1f"}
            ]
        }
    }
    
    html_zero_sales = vlc.vegalite_to_html(vl_spec=json.dumps(spec_zero_sales))
    with open(EXPORT_DIR / 'dq_zero_sales.html', 'w') as f:
        f.write(html_zero_sales)
    print("✓ Chart saved to dq_zero_sales.html")
    display(HTML(html_zero_sales))
else:
    print("WARNING: 'is_zero_sales' column not found")

## 6. Master Data Issues

### 6a. Customer Name Inconsistencies (Fuzzy Matching)

In [ ]:
# Simple detection of name inconsistencies by checking customer_number to customer_name mapping
if 'customer_number' in customer_data.columns and 'customer_name' in customer_data.columns:
    # Get unique customer_number to customer_name mappings
    cust_mapping = customer_data[['customer_number', 'customer_name']].drop_duplicates()
    
    # Find customer_numbers that map to multiple names (case-insensitive)
    cust_counts = cust_mapping.groupby('customer_number').size()
    duplicates = cust_counts[cust_counts > 1]
    
    print(f"Customer numbers with multiple name variations: {len(duplicates)}")
    print(f"Expected from PDF: ~8 cases")
    
    if len(duplicates) > 0:
        print(f"\nExamples of name inconsistencies:")
        for cust_num in list(duplicates.index)[:5]:
            names = cust_mapping[cust_mapping['customer_number'] == cust_num]['customer_name'].unique()
            print(f"  {cust_num}: {names}")
    
    # Create a summary table
    summary_table = []
    for cust_num in list(duplicates.index)[:10]:
        names = cust_mapping[cust_mapping['customer_number'] == cust_num]['customer_name'].unique()
        for i, name in enumerate(names):
            summary_table.append({
                "customer_number": cust_num,
                "variant_name": name,
                "issue": "Name Inconsistency (Same ID, Different Names)"
            })
    
    if summary_table:
        print(f"\nName inconsistency table:")
        df_inconsist = pd.DataFrame(summary_table)
        display(df_inconsist)
else:
    print("Note: customer_number or customer_name columns not found")

### 6b. No Unique Customer ID (Structural Issue)

In [ ]:
# Cardinality check: how many distinct values for customer identifier columns?
if all(col in customer_data.columns for col in ['customer_name', 'customer_group', 'customer_number']):
    distinct_names = customer_data['customer_name'].nunique()
    distinct_groups = customer_data['customer_group'].nunique()
    distinct_numbers = customer_data['customer_number'].nunique()
    total_rows = len(customer_data)
    
    print(f"Distinct customer identifiers (of {total_rows:,} rows):")
    print(f"  Distinct customer_name: {distinct_names:,}")
    print(f"  Distinct customer_group: {distinct_groups:,}")
    print(f"  Distinct customer_number: {distinct_numbers:,}")
    
    # Check ambiguity: how many customer_numbers map to multiple customer_groups?
    cust_num_to_groups = customer_data[['customer_number', 'customer_group']].drop_duplicates().groupby('customer_number').size()
    ambiguous_nums = (cust_num_to_groups > 1).sum()
    
    print(f"\nMaster data ambiguity:")
    print(f"  Customer numbers mapping to >1 customer_group: {ambiguous_nums}")
    print(f"  Ambiguity ratio: {(ambiguous_nums / distinct_numbers) * 100:.1f}%")
    
    # Summary for visualization
    card_data = [
        {"identifier": "Distinct Names", "count": distinct_names, "type": "Name"},
        {"identifier": "Distinct Groups", "count": distinct_groups, "type": "Group"},
        {"identifier": "Distinct Numbers", "count": distinct_numbers, "type": "Number"},
    ]
    
    spec_cardinality = {
        "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
        "title": "Master Data Ambiguity: Customer Identifier Cardinality",
        "width": 500,
        "height": 300,
        "data": {"values": card_data},
        "mark": "bar",
        "encoding": {
            "x": {"field": "identifier", "type": "nominal", "title": "Identifier"},
            "y": {"field": "count", "type": "quantitative", "title": "Distinct Count"},
            "color": {"value": BLUE},
            "tooltip": [
                {"field": "identifier", "type": "nominal", "title": "Field"},
                {"field": "count", "type": "quantitative", "title": "Distinct Values"}
            ]
        }
    }
    
    html_card = vlc.vegalite_to_html(vl_spec=json.dumps(spec_cardinality))
    with open(EXPORT_DIR / 'dq_customer_id_ambiguity.html', 'w') as f:
        f.write(html_card)
    print("✓ Chart saved to dq_customer_id_ambiguity.html")
    display(HTML(html_card))
else:
    print("Note: Required customer identifier columns not found")

## 7. Combined Impact Summary

In [ ]:
# Combined impact: what % of rows have ANY flag?
flag_cols_customer = [c for c in customer_data.columns if c.startswith('is_')]
flag_cols_product = [c for c in product_data.columns if c.startswith('is_')]

print(f"Customer flag columns: {flag_cols_customer}")
print(f"Product flag columns: {flag_cols_product}")

if flag_cols_customer:
    customer_data['has_any_flag'] = customer_data[flag_cols_customer].any(axis=1)
    flagged_cust = customer_data['has_any_flag'].sum()
    flagged_cust_pct = (flagged_cust / len(customer_data)) * 100
    clean_cust = len(customer_data) - flagged_cust
    print(f"\nCustomer data:")
    print(f"  Clean rows (no flags): {clean_cust:,} ({(clean_cust/len(customer_data))*100:.1f}%)")
    print(f"  Flagged rows (≥1 issue): {flagged_cust:,} ({flagged_cust_pct:.1f}%)")

if flag_cols_product:
    product_data['has_any_flag'] = product_data[flag_cols_product].any(axis=1)
    flagged_prod = product_data['has_any_flag'].sum()
    flagged_prod_pct = (flagged_prod / len(product_data)) * 100
    clean_prod = len(product_data) - flagged_prod
    print(f"\nProduct data:")
    print(f"  Clean rows (no flags): {clean_prod:,} ({(clean_prod/len(product_data))*100:.1f}%)")
    print(f"  Flagged rows (≥1 issue): {flagged_prod:,} ({flagged_prod_pct:.1f}%)")

# Visualization: stacked bar
impact_data = [
    {"table": "Customer", "status": "Clean", "count": clean_cust},
    {"table": "Customer", "status": "Has Issues", "count": flagged_cust},
    {"table": "Product", "status": "Clean", "count": clean_prod},
    {"table": "Product", "status": "Has Issues", "count": flagged_prod}
]

spec_impact = {
    "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
    "title": "Combined Impact: % of Data Rows with DQ Issues",
    "width": 500,
    "height": 300,
    "data": {"values": impact_data},
    "mark": "bar",
    "encoding": {
        "x": {"field": "table", "type": "nominal", "title": "Data Table"},
        "y": {"field": "count", "type": "quantitative", "title": "Row Count"},
        "color": {
            "field": "status",
            "type": "nominal",
            "scale": {"domain": ["Clean", "Has Issues"], "range": [GREEN, RED]}
        },
        "tooltip": [
            {"field": "table", "type": "nominal", "title": "Table"},
            {"field": "status", "type": "nominal", "title": "Status"},
            {"field": "count", "type": "quantitative", "title": "Count"}
        ]
    }
}

html_impact = vlc.vegalite_to_html(vl_spec=json.dumps(spec_impact))
with open(EXPORT_DIR / 'dq_combined_impact.html', 'w') as f:
    f.write(html_impact)
print("\n✓ Chart saved to dq_combined_impact.html")
display(HTML(html_impact))

## 8. Export Results

In [ ]:
# Create DQ report summary
dq_report = []

for idx, row in scorecard_df.iterrows():
    dq_report.append({
        'Issue': row['check'],
        'Data Table': row['table'],
        'Rows Affected': row['count'],
        'Percent Affected': f"{row['pct_affected']:.1f}%",
        'Severity': row['severity'],
        'Category': 'Missing Data' if 'missing' in row['check'] else ('Faulty Data' if 'negative' in row['check'] or 'zero' in row['check'] else 'Other')
    })

dq_report_df = pd.DataFrame(dq_report)

print(f"DQ Report Summary:")
print(dq_report_df.to_string(index=False))

# Save to CSV
csv_path = EXPORT_DIR / 'dq_report.csv'
dq_report_df.to_csv(csv_path, index=False)
print(f"\n✓ DQ report saved to CSV: {csv_path}")

# Save to XLSX
xlsx_path = EXPORT_DIR / 'dq_report.xlsx'
dq_report_df.to_excel(xlsx_path, index=False, sheet_name='Data Quality Issues')
print(f"✓ DQ report saved to XLSX: {xlsx_path}")

## ConclusionThis notebook successfully:1. Loaded customer and product data from **silver layer** (cleaned & flagged)2. Computed **DQ Scorecard** — overview of identified DQ issues with severity coloring3. Analyzed **Missing Data** issues:   - Buying Group L6 gaps (product)   - Gross Profit completeness (product)   - Booked Date (customer)   - Coverage gap across tables4. Analyzed **Faulty / Implausible Data**:   - Negative GP lines   - Cost-without-sales anomalies   - Zero/negative sales (returns/credits)5. Identified **Master Data Issues**:   - Customer name inconsistencies   - No unique customer ID (structural issue in data modeling)6. Computed **Combined Impact**: percentage of rows with DQ issues7. Generated interactive Vega-Lite HTML charts for each section8. Exported DQ report summary to CSV and XLSX**Key Takeaway:** The data reveals structural data quality issues. These issues require Phase-1 data requests and rule-agreement to address properly.